In [1]:
import os
import pickle
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC

In [2]:
def train_model(train_data, model_type):
    # Split dataset to train(80%) and test(20%) dataset
    train_data, test_data = train_test_split(train_data, test_size=0.2, random_state=42)

    X_train = train_data['content']
    y_train = train_data['labels']
    X_test = test_data['content']
    y_test = test_data['labels']

    # remove NaN, Null
    mask_train = X_train.notna() & y_train.notna()
    X_train = X_train[mask_train]
    y_train = y_train[mask_train]

    mask_test = X_test.notna() & y_test.notna()
    X_test = X_test[mask_test]
    y_test = y_test[mask_test]

    # Initialize TF-IDF Vectorizer and transform data
    tfidf_vectorizer = TfidfVectorizer(max_features=5000)
    X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
    X_test_tfidf = tfidf_vectorizer.transform(X_test)

    # Save TfidfVectorizer
    tfidt_path = os.path.join('process/model/trained_model', 'tfidf_vectorizer.pk')
    with open(tfidt_path, 'wb') as f:
        pickle.dump(tfidf_vectorizer, f)

    if model_type == 'bayes':
        print('Training bayes...')
        # Train Naive Bayes model
        model = MultinomialNB()
        model.fit(X_train_tfidf, y_train)

        # Test model with test_data
        y_pred = model.predict(X_test_tfidf)

        # Accuracy
        accuracy = accuracy_score(y_test, y_pred)
        print(f'Bayes Accuracy: {accuracy}')

        # Save model
        model_path = os.path.join('process/model/trained_model', 'naive_bayes_model.sav')
        with open(model_path, 'wb') as f:
            pickle.dump(model, f)
        print(f'Save bayes model at: {model_path}')
        return model_path
        
    
    if model_type == 'svm':
        print('Traning SVM...')
        # Train SVM model
        model = SVC()
        model.fit(X_train_tfidf, y_train)

        # Test model with test_data
        y_pred = model.predict(X_test_tfidf)

        # Accuracy
        accuracy = accuracy_score(y_test, y_pred)
        print(f'SVM Accuracy (SVM): {accuracy}')

        # Save model
        model_path = os.path.join('process/model/trained_model', 'svm_model.sav')
        with open(model_path, 'wb') as f:
            pickle.dump(model, f)
        print(f'Save SVM model at: {model_path}')
        return model_path

In [8]:
train_file_path = os.path.join('data', 'train_data.csv')
train_data = pd.read_csv(train_file_path)

test_file_path = os.path.join('data', 'test_data.csv')
test_data = pd.read_csv(test_file_path)

# Train model
bayes_model_path = train_model(train_data, 'bayes')
svm_model_path = train_model(train_data, 'svm')


Training bayes...
Bayes Accuracy: 0.6196717139585189
Save bayes model at: process/model/trained_model\naive_bayes_model.sav
Traning SVM...
SVM Accuracy (SVM): 0.6818933706578445
Save SVM model at: process/model/trained_model\svm_model.sav


In [9]:
_X_test = test_data['content']
_y_test = test_data['labels']

In [ ]:
# Load Naive Bayes model
with open(bayes_model_path, 'rb') as f:
    naive_bayes_model = pickle.load(f)
# Load SVM model
with open(svm_model_path, 'rb') as f:
    svm_model = pickle.load(f)
# Load TF-IDF Vectorizer
tfidt_path = os.path.join('process/model/trained_model', 'tfidf_vectorizer.pk')
with open(tfidt_path, 'rb') as f:
    tfidf_vectorizer = pickle.load(f)
    
# Transform data with TF-IDF Vectorizer
X_test_tfidf = tfidf_vectorizer.transform(_X_test)

In [ ]:
# Test Naive Bayes model
y_pred_nb = naive_bayes_model.predict(X_test_tfidf)
accuracy_nb = accuracy_score(_y_test, y_pred_nb)
print(f'Naive Bayes Accuracy: {accuracy_nb}')

Naive Bayes Accuracy: 0.5772727272727273


In [ ]:
# Test SVM model
y_pred_svm = svm_model.predict(X_test_tfidf)
accuracy_svm = accuracy_score(_y_test, y_pred_svm)
print(f'SVM Accuracy: {accuracy_svm}')

SVM Accuracy: 0.6488269794721407
